# Поиск изображения: матрица сходства

Дополнение к § 3.5. Числовой опыт показывает оси матрицы «снимки × описания» и влияние формулировки без скачивания CLIP.

## Матрица оценок
Строки соответствуют снимкам, столбцы — описаниям. Найдите лучший снимок для каждого текста.

In [ ]:
import numpy as np
images = np.array([[0.8,0.6],[0,1],[1,0]], dtype=float)
texts = np.array([[1,0],[0,1]], dtype=float)
images /= np.linalg.norm(images,axis=1,keepdims=True)
texts /= np.linalg.norm(texts,axis=1,keepdims=True)
scores = images @ texts.T
print(scores.round(3), scores.shape)
assert scores.shape == (3,2)

## Новый текст — новый порядок
Измените только вектор текста и сравните ранжирование. Это аналог изменения формулировки запроса.

In [ ]:
new_text = np.array([0.6,0.8])
new_text /= np.linalg.norm(new_text)
print(np.argsort(-(images @ new_text)).tolist())

## Нет подходящего кандидата
Ранжирование всё равно имеет лидера. Объясните, почему нужен отдельный способ отказаться от ответа.

In [ ]:
unrelated = np.array([-1.0,-1.0]); unrelated /= np.linalg.norm(unrelated)
print("Лидер среди кандидатов:", int(np.argmax(images @ unrelated)))

## Мини-проект с собственной коллекцией

Подготовьте шесть фотографий предметов без персональных данных и три английских описания. До запуска укажите, какие фотографии подходят каждому запросу, включая случай «ничего не подходит». Установите зависимости статьи, заполните пути и включите run_model.


In [ ]:
run_model = False
if run_model:
    from pathlib import Path
    import pandas as pd
    from PIL import Image
    import torch
    from transformers import CLIPModel, CLIPProcessor

    paths = [f"photo_{i}.jpg" for i in range(1, 7)]
    assert all(Path(name).is_file() for name in paths)
    images = [Image.open(name).convert("RGB") for name in paths]
    queries = [
        "a photo of a red figurine",
        "a photo of a 3D printer",
        "a photo of a blue figurine",
    ]
    model_id = "openai/clip-vit-base-patch32"
    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPModel.from_pretrained(model_id).eval()
    inputs = processor(
        text=queries, images=images, return_tensors="pt", padding=True
    )
    with torch.inference_mode():
        scores = model(**inputs).logits_per_image
    table = pd.DataFrame(scores.numpy(), index=paths, columns=queries)
    print(table.round(2).to_string())


## Проверяем порядок

Для каждого запроса выведите top-3 и сравните с ручной разметкой. Затем измените только формулировку одного запроса и пересчитайте матрицу. Запишите изменение мест, а не только оценку лидера. Если подходящего фото нет, отметьте, что CLIP всё равно поставил что-то первым.


In [ ]:
if run_model:
    for query in queries:
        print(query, table[query].sort_values(ascending=False)
              .head(3).index.tolist())


## Самопроверка




### 1. Что сравнивает CLIP в учебном поиске?

A. Два класса дерева.

B. Векторы изображения и текста.

C. Только размеры файлов.

D. Только цвета пикселей.




### 2. Чему равно сходство (1,0) и (0,8;0,6)?

A. 0.

B. 0,6.

C. 0,8.

D. 1.




### 3. Что означает первый результат среди трёх снимков?

A. Он обязательно подходит.

B. Он лидирует среди выбранных кандидатов.

C. Модель обучилась заново.

D. Остальные удалены.




### 4. Какова форма выхода при трёх изображениях и одном тексте?

A. 3×1.

B. 1×3.

C. 3×3.

D. 1×1.




### 5. Какова форма выхода при одном изображении и трёх описаниях?

A. 3×1.

B. 1×3.

C. 3×3.

D. 1×1.




### 6. Почему при изменении формулировки нужно оставить те же фотографии?

A. Чтобы связать изменение ранга с запросом.

B. Чтобы уменьшить число классов.

C. Чтобы обучить сеть.

D. Чтобы скрыть ошибки.




### 7. Можно ли назвать softmax по трём своим описаниям вероятностью истинного класса?

A. Да, всегда.

B. Только для красного.

C. Нет, доли зависят от списка вариантов.

D. Только на CPU.




### 8. Что делать, если подходящего снимка нет?

A. Объявить первый верным.

B. Допустить отказ и показать отсутствие подходящего.

C. Изменить веса без данных.

D. Удалить запрос.




### 9. Почему англоязычный CLIP нужно отдельно проверять на белорусских запросах?

A. Потому что оценки всегда нули.

B. Поддержка языка не следует из работы на английском.

C. Потому что нет букв.

D. Проверка запрещена.




### 10. Что такое попадание в top-3 в мини-проекте?

A. Хотя бы один подходящий снимок среди первых трёх.

B. Ровно три верных снимка.

C. Три новых веса.

D. Совпадение всех оценок.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Обе модальности получают представления.



**Решение**


**B.** Оценка получается из сопоставления изображения и описания.



### Вопрос 2

**Подсказка**


Оба вектора имеют длину 1.



**Решение**


**C.** Скалярное произведение равно 0,8.



### Вопрос 3

**Подсказка**


В коллекции может не быть подходящего кадра.



**Решение**


**B.** Ранжирование относительное и требует проверки.



### Вопрос 4

**Подсказка**


Строки соответствуют изображениям, столбцы — текстам.



**Решение**


**A.** Для каждой из трёх фотографий есть одна оценка запроса.



### Вопрос 5

**Подсказка**


Теперь меняется число текстов.



**Решение**


**B.** Одна строка содержит оценки трёх описаний.



### Вопрос 6

**Подсказка**


Меняйте одно условие за раз.



**Решение**


**A.** Иначе причина изменения порядка будет неясна.



### Вопрос 7

**Подсказка**


Добавление другого описания меняет знаменатель.



**Решение**


**C.** Это относительные доли внутри предложенного списка.



### Вопрос 8

**Подсказка**


Модель всё равно построит порядок.



**Решение**


**B.** Система должна учитывать возможность отсутствия совпадения.



### Вопрос 9

**Подсказка**


Область обучения влияет на применение.



**Решение**


**B.** Нужны отдельные примеры и оценка для другого языка.



### Вопрос 10

**Подсказка**


Сравните ручную разметку с первыми тремя позициями.



**Решение**


**A.** Это простая проверка, но малая коллекция не даёт общей оценки качества.



